In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from IPython.display import display


In [2]:
sns.set(
    context="notebook",
    rc={"figure.figsize": (12, 10)},
    palette=sns.color_palette("tab10", 10),
)

In [3]:
# Updated dataset paths with 'real' and 'synth' folders
datasets = {
    'Insurance': {
        'real': '../data/raw/data/insurance/real/insurance.csv',
        'synthetic': '../data/raw/data/insurance/synth/insurance_1M.csv'
    },
    'Diabetes': {
        'real': '../data/raw/data/diabetes/real/diabetes.csv',
        'synthetic': '../data/raw/data/diabetes/synth/diabetes_100K.csv'
    },
    'EPC': {
        'real': '../data/raw/data/epc/real/epc.csv',
        'synthetic': '../data/raw/data/epc/synth/epc_100K.csv'
    },
    'Income': {
        'real': '../data/raw/data/income/real/income.csv',
        'synthetic': '../data/raw/data/income/synth/income_100K.csv'
    },
    'Pollution': {
        'real': '../data/raw/data/pollution/real/pollution.csv',
        'synthetic': '../data/raw/data/pollution/synth/pollution_1000ID.csv'
    },
    'Wine': {
        'real': '../data/raw/data/wine/real/wine.csv',
        'synthetic': '../data/raw/data/wine/synth/wine_500K.csv'
    },
    'Macroenv': {
        'real': '../data/raw/data/macroenv/real/macroenv.csv',
        'synthetic': '../data/raw/data/macroenv/synth/macroenv_10KID.csv'
    }
}

In [4]:
# Ensure figures directory exists
os.makedirs('figures', exist_ok=True)

def separate_descriptive_stats(real_df, synth_df, numeric_cols):
    # Real data
    desc_real = real_df[numeric_cols].describe().T
    desc_real = desc_real.rename(columns={
        'count': 'Count',
        'mean': 'Mean',
        'std': 'Std Dev',
        'min': 'Min',
        '50%': 'Median',
        'max': 'Max'
    })
    desc_real = desc_real[['Count', 'Mean', 'Std Dev', 'Min', 'Median', 'Max']].T
    desc_real.columns.name = None
    desc_real.index.name = 'Statistic'
    desc_real = desc_real.round(2)
    desc_real.reset_index(inplace=True)

    # Synthetic data
    desc_synth = synth_df[numeric_cols].describe().T
    desc_synth = desc_synth.rename(columns={
        'count': 'Count',
        'mean': 'Mean',
        'std': 'Std Dev',
        'min': 'Min',
        '50%': 'Median',
        'max': 'Max'
    })
    desc_synth = desc_synth[['Count', 'Mean', 'Std Dev', 'Min', 'Median', 'Max']].T
    desc_synth.columns.name = None
    desc_synth.index.name = 'Statistic'
    desc_synth = desc_synth.round(2)
    desc_synth.reset_index(inplace=True)

    return desc_real, desc_synth


In [5]:
def save_histograms(real_df, synth_df, cols, dataset_name):
    for col in cols:
        # Combine data and drop NaNs
        combined_data = pd.concat([real_df[col], synth_df[col]], ignore_index=True).dropna()
        
        # Compute range and bins
        min_val = combined_data.min()
        max_val = combined_data.max()
        
        # Calculate bin edges
        bins = np.histogram_bin_edges(combined_data, bins=30, range=(min_val, max_val))
        
        plt.figure(figsize=(8, 4))
        
        sns.histplot(
            real_df[col].dropna(), 
            color='blue', 
            label='Real', 
            stat='density', 
            kde=True, 
            bins=bins, 
            alpha=0.5
        )
        
        sns.histplot(
            synth_df[col].dropna(), 
            color='red', 
            label='Synthetic', 
            stat='density', 
            kde=True, 
            bins=bins, 
            alpha=0.5
        )
        
        plt.title(f'{dataset_name} - {col} Distribution')
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig(f'figures/{dataset_name}_{col}_hist.png')
        plt.close()


In [6]:
def save_violin_plots(real_df, synth_df, cols, dataset_name):
    for col in cols:
        fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))  # Removed sharey=True

        # Real data violin plot
        sns.violinplot(
            y=real_df[col].dropna(),
            ax=axes[0],
            color='blue',
            inner='box',
            cut=0
        )
        axes[0].set_title('Real Data')
        axes[0].set_xlabel('')
        axes[0].set_ylabel(col)
        sns.despine(ax=axes[0], top=True, right=True)  # Remove spines

        # Synthetic data violin plot
        sns.violinplot(
            y=synth_df[col].dropna(),
            ax=axes[1],
            color='red',
            inner='box',
            cut=0
        )
        axes[1].set_title('Synthetic Data')
        axes[1].set_xlabel('')
        axes[1].set_ylabel('')
        sns.despine(ax=axes[1], top=True, right=True)  # Remove spines

        # Overall plot title
        fig.suptitle(f'{dataset_name} - {col} Violin Plots', fontsize=14)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(f'figures/{dataset_name}_{col}_violin.png')        
        plt.close()


In [7]:
def save_heatmap(df, cols, dataset_name, data_type):
    corr = df[cols].corr()
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
    plt.title(f'{dataset_name} - Correlation Heatmap ({data_type})')
    plt.tight_layout()
    plt.savefig(f'figures/{dataset_name}_{data_type}_heatmap.png')
    plt.close()




In [8]:
def anomaly_detection_all_numeric(real_df, synth_df):
    anomalies = {}
    
    numeric_cols_real = real_df.select_dtypes(include=['number']).columns
    numeric_cols_synth = synth_df.select_dtypes(include=['number']).columns    
    
    # Only proceed with columns present in both
    numeric_cols = set(numeric_cols_real).intersection(numeric_cols_synth)
    
    total_synth = len(synth_df)   
    
    for col in sorted(numeric_cols):
        real_min = real_df[col].min()
        real_max = real_df[col].max()
        synth_col = synth_df[col]
        
        below_min_count = (synth_col < real_min).sum()
        above_max_count = (synth_col > real_max).sum()
        neg_count = (synth_col < 0).sum()        
        
        if below_min_count > 0:
            anomalies[f'{col} below real min ({real_min})'] = (
                int(below_min_count),
                round((below_min_count / total_synth) * 100, 2)
            )
            
        if above_max_count > 0:
            anomalies[f'{col} above real max ({real_max})'] = (
                int(above_max_count),
                round((above_max_count / total_synth) * 100, 2)
            )
        
        # Only flag negatives if real min >= 0 (non-negative variable)
        if real_min >= 0 and neg_count > 0:
            anomalies[f'{col} negative values'] = (
                int(neg_count),
                round((neg_count / total_synth) * 100, 2)
            )
    
    return anomalies


In [9]:
x= pd.read_csv('../data/raw/data/diabetes/real/diabetes.csv')
y=pd.read_csv('../data/raw/data/diabetes/synth/diabetes_100k.csv')

In [10]:
cat_cols_real = x.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols_synth = y.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = list(set(cat_cols_real).intersection(cat_cols_synth))

In [11]:
for name, paths in datasets.items():
    print(f"\n\n=== Analyzing {name} Dataset ===")
    
    # Read data
    real_df = pd.read_csv(paths['real'])
    synth_df = pd.read_csv(paths['synthetic'])

    print(f"Dimension of real data: {real_df.shape}")
    print(f"Dimension of synthetic data: {synth_df.shape}")
    
    # Detect numeric columns
    numeric_cols_real = real_df.select_dtypes(include=['number']).columns.tolist()
    numeric_cols_synth = synth_df.select_dtypes(include=['number']).columns.tolist()
    numeric_cols = list(set(numeric_cols_real).intersection(set(numeric_cols_synth)))
    
    # ✅ Sort columns alphabetically
    numeric_cols.sort()
    
    print(f"Numeric columns detected ({len(numeric_cols)}): {numeric_cols}")
    
    # --- Metadata Comparison ---
    # Get dtypes for all columns in both datasets
    real_dtypes = real_df.dtypes.astype(str)
    synth_dtypes = synth_df.dtypes.astype(str)

    # Create comparison DataFrame
    meta_df = pd.DataFrame({
        'Real_dtype': real_dtypes,
        'Synthetic_dtype': synth_dtypes
    })
    meta_df.reset_index(inplace=True)
    meta_df.rename(columns={'index': 'Variable'}, inplace=True)

    # Add a check if types match
    meta_df['Type_Match'] = meta_df['Real_dtype'] == meta_df['Synthetic_dtype']
    
    # Check expanded ranges
    minmax_data = []
    for col in numeric_cols:
        real_min = real_df[col].min()
        real_max = real_df[col].max()
        synth_min = synth_df[col].min()
        synth_max = synth_df[col].max()
        expanded_range = (synth_min < real_min) or (synth_max > real_max)
        
        minmax_data.append({
            'Variable': col,
            'Real_Min': real_min,
            'Real_Max': real_max,
            'Synthetic_Min': synth_min,
            'Synthetic_Max': synth_max,
            'Range_Expanded': expanded_range
        })

    range_df = pd.DataFrame(minmax_data)
    
    # Display metadata comparison
    print("\n-- Metadata Comparison --")
    display(meta_df)
    
    # Display range comparison
    print("\n-- Range Comparison for Numeric Variables --")
    display(range_df)

    # Print textual metadata summary
    num_numeric = len(numeric_cols)
    num_binary = sum(real_df[col].nunique() == 2 for col in numeric_cols)
    print(f"\nMetadata Summary:")
    print(f"• Both datasets contain:")
    print(f"  o {num_numeric} numeric variables")
    print(f"  o {num_binary} binary outcome variable(s)")
    
    if meta_df['Type_Match'].all():
        print("• Variable types are identical (floats or integers).")
    else:
        mismatches = meta_df[~meta_df['Type_Match']]
        print("• WARNING: Some variable types differ between real and synthetic datasets!")
        print(mismatches.to_string(index=False))

    if range_df['Range_Expanded'].any():
        expanded_vars = range_df[range_df['Range_Expanded']]['Variable'].tolist()
        print(f"• The synthetic dataset expands the range of the following variable(s): {expanded_vars}")
    else:
        print("• The synthetic dataset does NOT expand variable ranges compared to the real data.")
    
    # descriptive statistics    
    desc_real, desc_synth = separate_descriptive_stats(real_df, synth_df, numeric_cols)

    print("\n-- Real Data Descriptive Statistics --")
    display(desc_real)            

    print("\n-- Synthetic Data Descriptive Statistics --")
    display(desc_synth)

    # --- Categorical Variable Distributions ---
    print("\n-- Categorical Variable Distributions --")

    # Detect categorical columns
    cat_cols_real = real_df.select_dtypes(include=['object', 'category']).columns.tolist()
    cat_cols_synth = synth_df.select_dtypes(include=['object', 'category']).columns.tolist()
    cat_cols = list(set(cat_cols_real).intersection(cat_cols_synth))
    cat_cols.sort()

    if cat_cols:
        for col in cat_cols:
            print(f"\n>> Variable: {col}")

            real_counts = real_df[col].value_counts()
            real_perc = real_df[col].value_counts(normalize=True).mul(100).round(2)
            
            synth_counts = synth_df[col].value_counts()
            synth_perc = synth_df[col].value_counts(normalize=True).mul(100).round(2)

            # Align categories
            idx = real_counts.index.union(synth_counts.index)
    
            real_counts = real_counts.reindex(idx, fill_value=0)
            real_perc   = real_perc.reindex(idx, fill_value=0.0)
            synth_counts = synth_counts.reindex(idx, fill_value=0)
            synth_perc   = synth_perc.reindex(idx, fill_value=0.0)
    
            # combine per metric
            counts = pd.concat([real_counts, synth_counts], axis=1, keys=['Real', 'Synthetic'])
            props  = pd.concat([real_perc, synth_perc], axis=1, keys=['Real', 'Synthetic'])
    
            # combine metrics with MultiIndex — first level = metric, second level = dataset
            combined = pd.concat([counts, props], axis=1, keys=['Count', 'Percentage(%)'])
    
            combined.index.name = col
            combined = combined.sort_index()          

            
            display(combined)
    else:
        print("No shared categorical variables found in both datasets.")

          
    # Anomaly detection
    anomalies = anomaly_detection_all_numeric(real_df, synth_df)
    print("\n-- Anomalies Detected Across All Numeric Columns --")
    if anomalies:
        anomalies_df = pd.DataFrame(
            [(k, v[0], v[1]) for k, v in anomalies.items()],
            columns=['Anomaly', 'Count', 'Percent']
        )
        display(anomalies_df)          
    else:
        print("No anomalies detected.")
    
    # Save histograms
    print("\n-- Generating Histograms --")
    save_histograms(real_df, synth_df, numeric_cols, name)

    # Violin Plots
    print("-- Generating Violin Plots --")
    save_violin_plots(real_df, synth_df, numeric_cols, name)
    
    # Save heatmaps
    print("-- Generating Heatmaps --")
    save_heatmap(real_df, numeric_cols, name, 'Real')
    save_heatmap(synth_df, numeric_cols, name, 'Synthetic')




=== Analyzing Insurance Dataset ===
Dimension of real data: (1338, 7)
Dimension of synthetic data: (1000000, 7)
Numeric columns detected (4): ['AGE', 'BMI', 'CHARGES', 'CHILDREN']

-- Metadata Comparison --


,Variable,Real_dtype,Synthetic_dtype,Type_Match
0,AGE,int64,float64,False
1,SEX,object,object,True
2,BMI,float64,float64,True
3,CHILDREN,int64,int64,True
4,SMOKER,object,object,True
5,REGION,object,object,True
6,CHARGES,float64,float64,True



-- Range Comparison for Numeric Variables --


,Variable,Real_Min,Real_Max,Synthetic_Min,Synthetic_Max,Range_Expanded
0,AGE,18.0000,64.00000,10.829939,73.049334,True
1,BMI,15.9600,53.13000,12.464947,57.995008,True
2,CHARGES,1121.8739,63770.42801,-2495.680369,65486.244230,True
3,CHILDREN,0.0000,5.00000,0.000000,5.000000,False



Metadata Summary:
• Both datasets contain:
  o 4 numeric variables
  o 0 binary outcome variable(s)
• WARNING: Some variable types differ between real and synthetic datasets!
Variable Real_dtype Synthetic_dtype  Type_Match
     AGE      int64         float64       False
• The synthetic dataset expands the range of the following variable(s): ['AGE', 'BMI', 'CHARGES']

-- Real Data Descriptive Statistics --


...


-- Synthetic Data Descriptive Statistics --


,Statistic,CPIAUCSL,CPIAUCSL_prev_state,GDPC1,GDPC1_prev_state,TERMCBAUTO48NS,TERMCBAUTO48NS_prev_state,UNRATE,UNRATE_prev_state,btc,...,doge_prev_state_last_1,eth,eth_prev_state_last_1,id,month,month_prev_state_last_1,usdt,usdt_prev_state_last_1,year,year_prev_state_last_1
0,Count,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,...,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00,30000.00
1,Mean,264.12,264.71,20640.70,20622.05,5.35,5.32,4.58,4.63,22.20,...,0.04,-10.97,795.96,601.50,6.81,6.75,-0.01,1.00,2019.44,2019.38
2,Std Dev,19.82,21.38,1036.75,1043.62,1.08,1.05,1.46,1.53,788.35,...,0.07,66.22,987.37,288.68,3.63,3.66,0.01,0.01,2.39,2.38
3,Min,238.40,237.34,18892.35,18892.25,4.00,4.00,3.43,3.43,-5032.33,...,0.00,-389.11,1.07,102.00,1.00,1.00,-0.20,0.98,2015.00,2015.00
4,Median,256.70,258.81,20595.27,20615.82,5.11,5.09,4.07,4.10,18.58,...,0.01,-9.51,321.04,601.50,7.00,7.00,-0.00,1.00,2019.00,2019.00
5,Max,305.08,308.85,22490.55,22490.50,8.51,8.51,12.92,13.29,5647.56,...,0.32,287.69,3975.42,1101.00,12.00,12.00,0.08,1.03,2024.00,2023.00



-- Categorical Variable Distributions --
No shared categorical variables found in both datasets.

-- Anomalies Detected Across All Numeric Columns --


,Anomaly,Count,Percent
0,id above real max (101),30000,100.0



-- Generating Histograms --
-- Generating Violin Plots --
-- Generating Heatmaps --
